<a href="https://colab.research.google.com/github/danielrodrges/Agente-IA/blob/main/aula_langgraph_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕸️ Aula Prática: Engenharia de Agentes com LangGraph

**Bem-vindos ao Nível Hard (e Robusto)!**

Até agora vimos:
1. **CrewAI:** Ótimo para orquestrar times (Alto nível).
2. **Agno:** Ótimo para assistentes com ferramentas (Baixo/Médio nível).

Hoje vamos de **LangGraph**. Ele é a ferramenta para quando você precisa de **Controle Total**.
Em vez de confiar que o LLM vai seguir um fluxo, nós vamos **desenhar** o fluxo como um grafo (Fluxograma) e forçar o agente a seguir caminhos específicos.

**Conceito Central:** State Machine (Máquina de Estados).
Imagine um grafo onde os **Nós** são funções (trabalho) e as **Arestas** são o caminho.

### O que vamos cobrir hoje:

| Seção | Tópico |
|--------|--------|
| 1-7 | Fundamentos: Estado, Nós, Arestas, Loop ReAct |
| 8 | Persistência e Memória (Checkpointers) |
| 9 | Human-in-the-Loop completo |
| 10 | Streaming de Tokens em tempo real |
| 11 | Time Travel ("Git para agentes") |
| 12 | Grafo Multi-Nó com lógica complexa |
| 13 | Multi-Agente com Supervisor |
| 14 | Estado Customizado e Reducers |
| 15 | Comparação final + Exercício |

---
## 1. Instalação e Configuração
O LangGraph roda em cima do LangChain. Vamos instalar ambos.

In [ ]:
!pip install -q -U langgraph langchain langchain-openai langchain-community duckduckgo-search ddgs

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Cole sua OpenAI API Key: ")

---
## 2. Definindo o Estado (The State)

No LangGraph, o "Estado" é uma estrutura de dados que passa de um nó para o outro.
É como se fosse uma **prancheta** que cada funcionário lê, anota algo novo e passa para o próximo.

Usamos `add_messages` como **reducer** — ele garante que as mensagens são sempre acumuladas (nunca sobrescritas).

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

print("✅ Estado definido! Cada nó vai receber e retornar 'messages'.")
print("O reducer 'add_messages' garante que mensagens novas são ADICIONADAS (nunca sobrescritas).")

---
## 3. Criando as Ferramentas e o Modelo
Vamos usar o DuckDuckGo como ferramenta e o GPT como cérebro.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchResults

tool = DuckDuckGoSearchResults()
tools = [tool]

model = ChatOpenAI(model="gpt-5-mini", reasoning_effort="minimal")
model_with_tools = model.bind_tools(tools)

print(f"✅ Modelo: gpt-5-mini | Ferramentas: {[t.name for t in tools]}")

---
## 4. Definindo os Nós (Nodes)

Vamos ter 2 nós principais:
1. **Agente (Reasoning):** O LLM pensa e decide o que fazer.
2. **Tool (Action):** Se o agente pediu uma ferramenta, nós a executamos aqui.

In [ ]:
from langgraph.prebuilt import ToolNode

def agent_node(state):
    """Nó do Agente: invoca o LLM com o histórico e retorna a resposta."""
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

print("✅ Nós definidos: 'agent_node' (cérebro) e 'tool_node' (braços)")

---
## 5. Definindo as Arestas (Edges) e o Grafo 🕸️

Aqui é a engenharia pura. Vamos desenhar o fluxo:
1. Começa no `agent`.
2. O `agent` decide.
3. **Condicional:**
    * Se ele pediu ferramenta → Vá para `tools`.
    * Se ele terminou (deu a resposta final) → Vá para `END`.
4. **Loop:** Se foi para `tools`, depois de rodar, volte para o `agent` (para ele ler o resultado).

In [ ]:
from langgraph.graph import StateGraph, END

def should_continue(state):
    """Se a última mensagem tem tool_calls, continua. Senão, finaliza."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "continue"
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"continue": "tools", "end": END},
)

workflow.add_edge("tools", "agent")

app = workflow.compile()
print("✅ Grafo compilado com sucesso!")

---
## 6. Visualizando o Grafo
Uma das coisas mais legais do LangGraph é poder visualizar a lógica.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Instale 'grandalf' para ver o gráfico. O código funciona sem ele!")

---
## 7. Executando o Agente

Agora vamos testar. Observe que o fluxo vai entrar em **Loop** automaticamente se precisar.
Perguntaremos algo que exige pesquisa.

In [ ]:
from langchain_core.messages import HumanMessage

pergunta = "Qual foi o vencedor da Copa Libertadores e quem foi o vice em 2026?"

inputs = {"messages": [HumanMessage(content=pergunta)]}

print(f"🤖 Usuário: {pergunta}\n")
print("=" * 60)

for step, output in enumerate(app.stream(inputs), 1):
    for node_name, value in output.items():
        msg = value["messages"][0]
        print(f"\n🔹 Passo {step} — Nó: '{node_name}'")
        print("-" * 40)
        if node_name == "agent":
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 Chamou ferramenta: {tc['name']}")
                    print(f"   📝 Query: {tc['args']}")
            else:
                print(f"   💡 Resposta Final:\n\n{msg.content}")
        elif node_name == "tools":
            print(f"   📦 Ferramenta retornou {len(msg.content)} caracteres de dados.")

print("\n" + "=" * 60)
print("✅ Execução finalizada!")

---
---
# Parte 2: LangGraph avançado

A partir daqui vamos explorar features que tornam o LangGraph **único** entre os frameworks de agentes.

---
## 8. Persistência e Memória (Checkpointers) 🧠

### O Problema
Nosso agente da seção 7 é **stateless** — ele esquece tudo após cada chamada.
Se você perguntar "O que eu disse antes?", ele não vai saber.

### A Solução: Checkpointers
O LangGraph salva um **snapshot** (checkpoint) do estado após cada nó executado.
Pense nisso como um **"auto-save" de videogame** — a cada passo, o estado é salvo.

Cada conversa é identificada por um `thread_id`, como um número de sessão.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

app_with_memory = workflow.compile(checkpointer=memory)

print("✅ Grafo recompilado COM memória (MemorySaver)!")
print("Agora cada conversa é identificada por um 'thread_id'.")

In [ ]:
config_thread1 = {"configurable": {"thread_id": "conversa-1"}}

print("--- Mensagem 1 (conversa-1) ---")
response1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="Oi! Meu nome é Filipe e eu trabalho com IA.")]},
    config_thread1,
)
print(response1["messages"][-1].content)

print("\n--- Mensagem 2 (conversa-1) ---")
response2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="Qual é o meu nome e com o que eu trabalho?")]},
    config_thread1,
)
print(response2["messages"][-1].content)

### Isolamento entre threads
Cada `thread_id` é uma conversa independente. Vamos provar:

In [ ]:
config_thread2 = {"configurable": {"thread_id": "conversa-2"}}

print("--- Mensagem na conversa-2 (thread diferente) ---")
response3 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="Qual é o meu nome?")]},
    config_thread2,
)
print(response3["messages"][-1].content)
print("\n👆 Ele NÃO sabe! Porque 'conversa-2' nunca viu a informação do Filipe.")

---
## 9. Human-in-the-Loop Completo ✋

No CrewAI, fazer um humano aprovar uma ação é difícil. No LangGraph, é **nativo**.

O fluxo completo:
1. O agente decide chamar uma ferramenta
2. O grafo **pausa** antes de executar (`interrupt_before`)
3. O humano **inspeciona** o estado
4. O humano **aprova** (continua) ou **rejeita** (modifica o estado)

Para isso funcionar, precisamos de um **checkpointer** (para salvar o estado no ponto de pausa).

In [ ]:
memory_hitl = MemorySaver()

app_hitl = workflow.compile(
    checkpointer=memory_hitl,
    interrupt_before=["tools"],
)

print("✅ Grafo compilado com Human-in-the-Loop!")
print("O grafo vai PAUSAR antes de executar qualquer ferramenta.")

display(Image(app_hitl.get_graph().draw_mermaid_png()))

In [ ]:
config_hitl = {"configurable": {"thread_id": "hitl-demo"}}

print("🚀 Enviando pergunta que vai exigir ferramenta...\n")

for output in app_hitl.stream(
    {"messages": [HumanMessage(content="Pesquise: qual o PIB do Brasil em 2025?")]},
    config_hitl,
):
    for node_name, value in output.items():
        print(f"🔹 Nó '{node_name}' executou.")
        if node_name == "agent" and value["messages"][0].tool_calls:
            for tc in value["messages"][0].tool_calls:
                print(f"   🔧 Quer chamar: {tc['name']}")
                print(f"   📝 Args: {tc['args']}")

print("\n⏸️  GRAFO PAUSADO! Esperando aprovação humana...")

### Inspecionando o Estado
Antes de aprovar, podemos ver exatamente o que o agente quer fazer:

In [ ]:
state = app_hitl.get_state(config_hitl)

print("Estado atual do grafo:")
print(f"  Próximo nó a executar: {state.next}")
print(f"  Nº de mensagens no estado: {len(state.values['messages'])}")

last_msg = state.values["messages"][-1]
if last_msg.tool_calls:
    print(f"\n  🔍 Ferramenta pendente:")
    for tc in last_msg.tool_calls:
        print(f"     Nome: {tc['name']}")
        print(f"     Args: {tc['args']}")

### Opção A: Aprovar (continuar execução)

In [ ]:
print("✅ Humano aprovou! Continuando execução...\n")

for output in app_hitl.stream(None, config_hitl):
    for node_name, value in output.items():
        msg = value["messages"][0]
        print(f"🔹 Nó '{node_name}' executou.")
        if node_name == "agent" and not msg.tool_calls:
            print(f"\n💡 Resposta Final:\n{msg.content}")

print("\n✅ Execução finalizada!")

### Opção B: Rejeitar (modificar o estado)

E se o humano não quiser que a ferramenta rode?
Podemos **injetar uma resposta manual** no estado e deixar o agente continuar com ela.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

memory_hitl2 = MemorySaver()
app_hitl2 = workflow.compile(checkpointer=memory_hitl2, interrupt_before=["tools"])

config_hitl2 = {"configurable": {"thread_id": "hitl-reject"}}

print("🚀 Enviando pergunta...\n")
for output in app_hitl2.stream(
    {"messages": [HumanMessage(content="Pesquise: quais foram as maiores aquisições de empresas de IA em 2025?")]},
    config_hitl2,
):
    for node_name, value in output.items():
        if node_name == "agent" and value["messages"][0].tool_calls:
            print(f"⏸️  Pausou. Agente quer pesquisar: {value['messages'][0].tool_calls[0]['args']}")

print("\n❌ Humano REJEITOU a pesquisa! Injetando resposta manual...\n")

current_state = app_hitl2.get_state(config_hitl2)
last_msg = current_state.values["messages"][-1]

if not hasattr(last_msg, "tool_calls") or not last_msg.tool_calls:
    raise RuntimeError(
        f"O agente não chamou nenhuma ferramenta — respondeu direto. "
        f"Última mensagem: {last_msg.content[:100]}..."
    )

tool_call_id = last_msg.tool_calls[0]["id"]

app_hitl2.update_state(
    config_hitl2,
    {
        "messages": [
            ToolMessage(
                content="O humano informou: A maior aquisição de IA em 2025 foi a compra da Character.AI pela Google. Não precisa pesquisar mais.",
                tool_call_id=tool_call_id,
            )
        ]
    },
    as_node="tools",
)

print("🔄 Continuando com a resposta manual...\n")

for output in app_hitl2.stream(None, config_hitl2):
    for node_name, value in output.items():
        msg = value["messages"][0]
        if node_name == "agent" and not msg.tool_calls:
            print(f"💡 Resposta Final (baseada na info manual):\n{msg.content}")

---
## 10. Streaming de Tokens em Tempo Real 🌊

Até agora usamos `stream()` para ver a execução **nó a nó**.
Mas em produção queremos ver os **tokens chegando um a um** (como o ChatGPT faz).

O LangGraph oferece dois modos de streaming:

| Modo | O que mostra |
|------|-------------|
| `stream_mode="updates"` | Um evento por nó (padrão) |
| `stream_mode="messages"` | Cada token do LLM conforme é gerado |

In [ ]:
from langchain_core.messages import AIMessageChunk

memory_stream = MemorySaver()
app_stream = workflow.compile(checkpointer=memory_stream)

config_stream = {"configurable": {"thread_id": "stream-demo"}}

print("Streaming token a token (como o ChatGPT):\n")
print("─" * 50)

for event, metadata in app_stream.stream(
    {"messages": [HumanMessage(content="Explique em 10 frases o que é Machine Learning.")]},
    config_stream,
    stream_mode="messages",
):
    if isinstance(event, AIMessageChunk) and event.content:
        print(event.content, end="", flush=True)

print("\n" + "─" * 50)
print("\n✅ Streaming completo!")

### Comparando com stream_mode="updates" (padrão)
Aqui você só recebe o resultado completo de cada nó:

In [ ]:
config_stream2 = {"configurable": {"thread_id": "stream-updates-demo"}}

print("Streaming por nó (updates):\n")
print("─" * 50)

for output in app_stream.stream(
    {"messages": [HumanMessage(content="O que é Deep Learning? Responda em 10 frases.")]},
    config_stream2,
    stream_mode="updates",
):
    for node_name, value in output.items():
        msg = value["messages"][0]
        print(f"\n[🔹 Nó: {node_name}]")
        if hasattr(msg, "content") and msg.content:
            print(msg.content)

print("\n" + "─" * 50)
print("\n👆 Perceba: o texto chegou tudo de uma vez, não token a token.")

---
## 11. Time Travel — "Git para Agentes de IA" ⏪

Lembra que o checkpointer salva um snapshot a cada passo?
Isso nos dá um **histórico completo** da execução.

Podemos:
- **Replay**: Voltar a um ponto anterior e re-executar
- **Fork**: Modificar o estado de um ponto anterior e explorar um caminho alternativo

Pense nisso como `git log`, `git checkout` e `git branch` para agentes!

In [ ]:
memory_tt = MemorySaver()
app_tt = workflow.compile(checkpointer=memory_tt)

config_tt = {"configurable": {"thread_id": "time-travel-demo"}}

print("Executando conversa com 3 mensagens para gerar histórico...\n")

app_tt.invoke(
    {"messages": [HumanMessage(content="Oi! Me chamo Ana.")]},
    config_tt,
)
print("✅ Mensagem 1 enviada.")

app_tt.invoke(
    {"messages": [HumanMessage(content="Eu moro em São Paulo.")]},
    config_tt,
)
print("✅ Mensagem 2 enviada.")

app_tt.invoke(
    {"messages": [HumanMessage(content="O que você sabe sobre mim?")]},
    config_tt,
)
print("✅ Mensagem 3 enviada.")

print("\n📚 Conversa completa! Agora vamos explorar o histórico...")

In [ ]:
print("📜 Histórico de Checkpoints (como 'git log'):\n")

checkpoints = list(app_tt.get_state_history(config_tt))

for i, state in enumerate(checkpoints):
    n_msgs = len(state.values.get("messages", []))
    next_node = state.next
    checkpoint_id = state.config["configurable"]["checkpoint_id"]
    print(f"  [{i}] Checkpoint: {checkpoint_id[:12]}... | Mensagens: {n_msgs} | Próximo nó: {next_node}")

print(f"\n📊 Total de checkpoints: {len(checkpoints)}")

### Fork: Viajando no tempo e mudando a história

Vamos pegar um checkpoint antigo (quando a Ana ainda não tinha dito onde mora)
e criar uma "realidade alternativa" onde ela mora no Rio:

In [ ]:
old_checkpoint = checkpoints[-3] if len(checkpoints) >= 3 else checkpoints[-1]
old_config = old_checkpoint.config

n_msgs_old = len(old_checkpoint.values.get("messages", []))
print(f"⏪ Voltando ao checkpoint com {n_msgs_old} mensagem(ns)...\n")

print("Mensagens naquele ponto:")
for msg in old_checkpoint.values.get("messages", []):
    role = "Usuário" if msg.type == "human" else "Agente"
    print(f"  [{role}]: {msg.content[:80]}..." if len(msg.content) > 80 else f"  [{role}]: {msg.content}")

print("\n🔀 Criando realidade alternativa (fork): Ana agora mora no Rio!\n")

fork_response = app_tt.invoke(
    {"messages": [HumanMessage(content="Na verdade, eu moro no Rio de Janeiro! O que você sabe sobre mim?")]},
    old_config,
)
print(f"💡 Resposta na timeline alternativa:\n{fork_response['messages'][-1].content}")

---
---
# Parte 3: Padrões Avançados

---
## 12. Grafo Multi-Nó com Lógica Complexa 🎯

Até agora nosso grafo tinha 2 nós (agent + tools). Mas o verdadeiro poder do LangGraph
é construir **pipelines com múltiplas etapas controladas**.

Vamos construir um agente de pesquisa em 3 etapas:

```
[Planner] → [Researcher] → [Writer] → END
                  ↑              │
                  └──────────────┘
           (se dados insuficientes)
```

- **Planner**: Recebe a pergunta e gera um plano de pesquisa
- **Researcher**: Executa buscas usando ferramentas
- **Writer**: Sintetiza os resultados. Se insuficientes, manda de volta ao Researcher

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    plan: str
    research_data: str
    revision_count: int

llm = ChatOpenAI(model="gpt-5-mini", reasoning_effort="minimal")
llm_with_tools = llm.bind_tools(tools)

print("✅ Estado customizado com 4 campos: messages, plan, research_data, revision_count")

In [ ]:
from langgraph.prebuilt import ToolNode

def planner_node(state):
    """Gera um plano de pesquisa com base na pergunta do usuário."""
    user_msg = state["messages"][-1].content
    response = llm.invoke([
        SystemMessage(content="Você é um planejador de pesquisa. Dado um tópico, gere um plano conciso com 2-3 pontos de pesquisa. Responda APENAS com o plano, sem introdução."),
        HumanMessage(content=f"Crie um plano de pesquisa para: {user_msg}"),
    ])
    return {"plan": response.content, "revision_count": 0}

def researcher_node(state):
    """Usa ferramentas para pesquisar com base no plano."""
    plan = state.get("plan", "")
    existing_data = state.get("research_data", "")

    prompt = f"Pesquise informações sobre este plano:\n{plan}"
    if existing_data:
        prompt += f"\n\nDados já coletados (insuficientes, busque mais):\n{existing_data}"

    response = llm_with_tools.invoke([
        SystemMessage(content="Você é um pesquisador. Use a ferramenta de busca para encontrar dados relevantes."),
        HumanMessage(content=prompt),
    ])
    return {"messages": [response]}

def researcher_tools_node(state):
    """Executa as ferramentas chamadas pelo researcher."""
    return ToolNode(tools).invoke(state)

def collect_research_node(state):
    """Coleta os resultados das ferramentas e salva em research_data."""
    tool_messages = [
        msg for msg in state["messages"]
        if hasattr(msg, "type") and msg.type == "tool"
    ]
    new_data = "\n".join(msg.content for msg in tool_messages[-3:])
    existing = state.get("research_data", "")
    return {"research_data": existing + "\n" + new_data if existing else new_data}

def writer_node(state):
    """Sintetiza os dados de pesquisa em uma resposta final."""
    plan = state.get("plan", "")
    data = state.get("research_data", "Nenhum dado encontrado.")
    user_msg = state["messages"][0].content

    response = llm.invoke([
        SystemMessage(
            content="Você é um escritor. Sintetize os dados de pesquisa em uma resposta clara e completa. "
            "Se os dados forem completamente insuficientes para responder bem, comece sua resposta com 'INSUFICIENTE:' e explique o que falta. "
            "Caso contrário, escreva a resposta final. Não escreva 'INSUFICIENTE:' se houver dados suficientes."
        ),
        HumanMessage(content=f"Pergunta: {user_msg}\n\nPlano: {plan}\n\nDados coletados:\n{data}"),
    ])
    return {"messages": [response]}

def writer_decision(state):
    """Decide se a resposta está boa ou precisa de mais pesquisa."""
    last_msg = state["messages"][-1].content
    revision_count = state.get("revision_count", 0)

    if last_msg.startswith("INSUFICIENTE:") and revision_count < 2:
        return "needs_more_research"
    return "done"

def increment_revision(state):
    """Incrementa o contador de revisões."""
    return {"revision_count": state.get("revision_count", 0) + 1}

print("✅ Todos os nós definidos!")

In [ ]:
def researcher_should_continue(state):
    """Verifica se o researcher chamou ferramentas."""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "use_tools"
    return "skip_tools"

research_workflow = StateGraph(ResearchState)

research_workflow.add_node("planner", planner_node)
research_workflow.add_node("researcher", researcher_node)
research_workflow.add_node("researcher_tools", researcher_tools_node)
research_workflow.add_node("collect_research", collect_research_node)
research_workflow.add_node("writer", writer_node)
research_workflow.add_node("increment_revision", increment_revision)

research_workflow.set_entry_point("planner")
research_workflow.add_edge("planner", "researcher")

research_workflow.add_conditional_edges(
    "researcher",
    researcher_should_continue,
    {"use_tools": "researcher_tools", "skip_tools": "writer"},
)
research_workflow.add_edge("researcher_tools", "collect_research")
research_workflow.add_edge("collect_research", "writer")

research_workflow.add_conditional_edges(
    "writer",
    writer_decision,
    {"needs_more_research": "increment_revision", "done": END},
)
research_workflow.add_edge("increment_revision", "researcher")

research_app = research_workflow.compile()

print("✅ Grafo de pesquisa multi-nó compilado!")
display(Image(research_app.get_graph().draw_mermaid_png()))

In [ ]:
print("🚀 Executando pipeline de pesquisa...\n")
print("=" * 60)

research_input = {
    "messages": [HumanMessage(content="Quais são as principais diferenças entre LangGraph e CrewAI para construção de agentes?")],
    "plan": "",
    "research_data": "",
    "revision_count": 0,
}

for output in research_app.stream(research_input):
    for node_name, value in output.items():
        print(f"\n🔹 Nó: '{node_name}'")
        print("-" * 40)
        if "plan" in value and value["plan"]:
            print(f"   📋 Plano gerado:\n{value['plan'][:200]}...")
        if "messages" in value:
            msg = value["messages"][0] if isinstance(value["messages"], list) else value["messages"]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print(f"   🔧 Pesquisando: {msg.tool_calls[0]['args']}")
            elif hasattr(msg, "content") and msg.content and node_name == "writer":
                print(f"   ✍️ Resposta do Writer:\n{msg.content[:300]}...")

print("\n" + "=" * 60)
print("✅ Pipeline de pesquisa finalizado!")

---
## 13. Multi-Agente com Supervisor 👑

O padrão **Supervisor** é quando um nó central (o "chefe") decide qual agente especialista chamar.

Vamos criar:
- **Supervisor**: Decide quem trabalha
- **Pesquisador**: Busca informações na web
- **Calculador**: Faz contas e análises numéricas
- **Escritor**: Produz textos finais

```
         ┌──────────────┐
         │  Supervisor  │
         └────┬────┬────┘
              │    │
    ┌───────┘    │    ┌───────┐
    │              │    │
 Pesquisador  Calculador  Escritor
```

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, Field

class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str

AGENTS = ["pesquisador", "calculador", "escritor"]

class SupervisorDecision(BaseModel):
    """Decisão do supervisor sobre qual agente chamar."""
    next: Literal["pesquisador", "calculador", "escritor", "FINISH"] = Field(
        description="O próximo agente a ser chamado, ou FINISH se a tarefa está completa."
    )

supervisor_llm = ChatOpenAI(model="gpt-5-mini", reasoning_effort="minimal")
supervisor_structured = supervisor_llm.with_structured_output(SupervisorDecision)

def supervisor_node(state):
    """O Supervisor analisa o estado e decide qual agente chamar (ou finalizar)."""
    system_prompt = (
        "Você é um supervisor que coordena 3 agentes especializados:\n"
        "- 'pesquisador': busca informações e dados\n"
        "- 'calculador': faz cálculos e análises numéricas\n"
        "- 'escritor': redige textos e respostas finais\n\n"
        "Analise a conversa e decida o próximo passo.\n"
        "Cada agente só precisa ser chamado UMA VEZ. Se o agente já respondeu, passe para o próximo.\n"
        "Fluxo típico: pesquisador → calculador → escritor → FINISH\n"
        "Responda com FINISH quando o escritor já tiver produzido a resposta final."
    )

    decision = supervisor_structured.invoke(
        [SystemMessage(content=system_prompt)] + state["messages"]
    )

    next_agent = decision.next
    return {"next_agent": next_agent, "messages": [AIMessage(content=f"Próximo: {next_agent}")]}

def pesquisador_node(state):
    """Agente especialista em pesquisa (sem ferramentas, usa conhecimento do LLM)."""
    response = llm.invoke([
        SystemMessage(content="Você é um pesquisador. Forneça dados e informações relevantes sobre o que foi pedido. Seja conciso e factual. Responda com os dados solicitados."),
        *state["messages"],
    ])
    return {"messages": [HumanMessage(content=f"[Pesquisador]: {response.content}")]}

def calculador_node(state):
    """Agente especialista em cálculos."""
    response = llm.invoke([
        SystemMessage(content="Você é um calculador. Faça cálculos precisos e apresente os números de forma clara."),
        *state["messages"],
    ])
    return {"messages": [HumanMessage(content=f"[Calculador]: {response.content}")]}

def escritor_node(state):
    """Agente especialista em escrita."""
    response = llm.invoke([
        SystemMessage(content="Você é um escritor. Sintetize todas as informações disponíveis em uma resposta final clara e bem estruturada."),
        *state["messages"],
    ])
    return {"messages": [HumanMessage(content=f"[Escritor]: {response.content}")]}

print("✅ Supervisor e 3 agentes especialistas definidos!")

In [ ]:
def route_supervisor(state):
    next_agent = state.get("next_agent", "FINISH")
    if next_agent == "FINISH":
        return "end"
    return next_agent

supervisor_workflow = StateGraph(SupervisorState)

supervisor_workflow.add_node("supervisor", supervisor_node)
supervisor_workflow.add_node("pesquisador", pesquisador_node)
supervisor_workflow.add_node("calculador", calculador_node)
supervisor_workflow.add_node("escritor", escritor_node)

supervisor_workflow.set_entry_point("supervisor")

supervisor_workflow.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "pesquisador": "pesquisador",
        "calculador": "calculador",
        "escritor": "escritor",
        "end": END,
    },
)

supervisor_workflow.add_edge("pesquisador", "supervisor")
supervisor_workflow.add_edge("calculador", "supervisor")
supervisor_workflow.add_edge("escritor", "supervisor")

supervisor_app = supervisor_workflow.compile()

print("✅ Grafo Supervisor compilado!")
display(Image(supervisor_app.get_graph().draw_mermaid_png()))

In [ ]:
print("🚀 Testando o Supervisor...\n")
print("=" * 60)

supervisor_input = {
    "messages": [
        HumanMessage(content="Descubra a população do Brasil e do México, calcule a diferença percentual entre eles, e escreva um parágrafo comparativo.")
    ],
    "next_agent": "",
}

for output in supervisor_app.stream(supervisor_input, {"recursion_limit": 25}):
    for node_name, value in output.items():
        print(f"\n🔹 Nó: '{node_name}'")
        print("-" * 40)
        if node_name == "supervisor":
            next_ag = value.get("next_agent", "?")
            print(f"   👑 Supervisor decidiu: → {next_ag}")
        else:
            msg = value["messages"][-1] if isinstance(value.get("messages"), list) else None
            if msg:
                content = msg.content if hasattr(msg, "content") else str(msg)
                preview = content[:200] + "..." if len(content) > 200 else content
                print(f"   {preview}")

print("\n" + "=" * 60)
print("✅ Supervisor finalizou!")

---
## 14. Estado Customizado e Reducers 🧩

Até agora vimos estados simples (só mensagens). Mas o LangGraph permite estados **ricos e tipados**.

### O que são Reducers?

Quando dois nós retornam o mesmo campo, o reducer decide **como combinar** os valores:

| Reducer | Comportamento |
|---------|---------------|
| `add_messages` | Acumula mensagens (nunca sobrescreve) |
| `operator.add` | Concatena listas |
| Nenhum (padrão) | Sobrescreve com o último valor |
| Função custom | Você decide a lógica |

Vamos criar um estado com múltiplos campos tipados e um reducer customizado.

In [ ]:
import operator

def max_reducer(current, new):
    """Reducer que mantém sempre o MAIOR valor."""
    if current is None:
        return new
    return max(current, new)

class AnalysisState(TypedDict):
    messages: Annotated[list, add_messages]
    topics_found: Annotated[list[str], operator.add]   # Acumula tópicos
    confidence_score: Annotated[float, max_reducer]     # Mantém o maior score
    analysis_step: str                                  # Sobrescreve (sem reducer)
    iteration: int                                      # Sobrescreve

print("✅ Estado customizado com 5 campos e 3 tipos de reducer!")
print()
print("Campos:")
print("  messages        → add_messages (acumula)")
print("  topics_found    → operator.add (concatena listas)")
print("  confidence_score → max_reducer (mantém o maior)")
print("  analysis_step   → sem reducer (sobrescreve)")
print("  iteration       → sem reducer (sobrescreve)")

In [ ]:
def discovery_node(state):
    """Descobre tópicos relevantes sobre a pergunta."""
    user_msg = state["messages"][0].content
    response = llm.invoke([
        SystemMessage(content="Extraia 3 tópicos-chave da pergunta. Responda APENAS com os tópicos separados por vírgula."),
        HumanMessage(content=user_msg),
    ])
    topics = [t.strip() for t in response.content.split(",")]
    return {
        "topics_found": topics,
        "confidence_score": 0.3,
        "analysis_step": "discovery",
        "iteration": 1,
    }

def deep_analysis_node(state):
    """Analisa cada tópico com mais profundidade."""
    topics = state["topics_found"]
    response = llm.invoke([
        SystemMessage(content="Para cada tópico, dê um insight de 1 frase."),
        HumanMessage(content=f"Tópicos: {', '.join(topics)}"),
    ])
    new_topics = [f"insight_{t}" for t in topics[:2]]
    return {
        "messages": [response],
        "topics_found": new_topics,
        "confidence_score": 0.8,
        "analysis_step": "deep_analysis",
        "iteration": 2,
    }

def summary_node(state):
    """Gera o resumo final."""
    return {
        "messages": [HumanMessage(content=f"Análise completa. Tópicos: {state['topics_found']}. Score: {state['confidence_score']}")],
        "confidence_score": 0.95,
        "analysis_step": "summary",
        "iteration": 3,
    }

analysis_workflow = StateGraph(AnalysisState)
analysis_workflow.add_node("discovery", discovery_node)
analysis_workflow.add_node("deep_analysis", deep_analysis_node)
analysis_workflow.add_node("summary", summary_node)

analysis_workflow.set_entry_point("discovery")
analysis_workflow.add_edge("discovery", "deep_analysis")
analysis_workflow.add_edge("deep_analysis", "summary")
analysis_workflow.add_edge("summary", END)

analysis_app = analysis_workflow.compile()

print("✅ Pipeline de análise compilado!")
display(Image(analysis_app.get_graph().draw_mermaid_png()))

In [ ]:
print("🚀 Executando pipeline de análise com estado customizado...\n")

analysis_input = {
    "messages": [HumanMessage(content="Como a inteligência artificial impacta a saúde, educação e meio ambiente?")],
    "topics_found": [],
    "confidence_score": 0.0,
    "analysis_step": "",
    "iteration": 0,
}

for output in analysis_app.stream(analysis_input):
    for node_name, value in output.items():
        print(f"🔹 Nó: '{node_name}'")
        if "topics_found" in value:
            print(f"   Tópicos encontrados: {value['topics_found']}")
        if "confidence_score" in value:
            print(f"   Score de confiança: {value['confidence_score']}")
        if "analysis_step" in value:
            print(f"   Etapa: {value['analysis_step']}")
        if "iteration" in value:
            print(f"   Iteração: {value['iteration']}")
        print()

final_state = analysis_app.invoke(analysis_input)
print("=" * 60)
print("Estado Final:")
print(f"  Tópicos (acumulados via operator.add): {final_state['topics_found']}")
print(f"  Score (via max_reducer — só cresce): {final_state['confidence_score']}")
print(f"  Etapa (sobrescrita): {final_state['analysis_step']}")
print(f"  Iteração (sobrescrita): {final_state['iteration']}")

---
---
# Parte 4: Encerramento

---
## 15. Comparação Final: CrewAI vs Agno vs LangGraph

| Aspecto | CrewAI | Agno | LangGraph |
|---------|--------|------|-----------|
| **Nível de controle** | Alto nível (crews e roles) | Médio (assistentes) | Baixo nível (controle total) |
| **Persistência** | Limitada | Manual | Nativa (checkpointers) |
| **Human-in-the-Loop** | Callbacks básicos | Manual | Nativo (interrupt + update_state) |
| **Multi-Agente** | Crews com roles | Não nativo | Subgraphs + Supervisor |
| **Streaming** | Limitado | Básico | Granular (tokens/eventos/nós) |
| **Time Travel** | Não | Não | Nativo (replay + fork) |
| **Visualização** | Não nativa | Não nativa | Nativa (Mermaid) |
| **Estado Customizado** | Limitado | Dicionários | TypedDict + Reducers |
| **Quando usar** | Times de agentes com roles claros | Assistentes com tools | Fluxos complexos que exigem controle |

### Resumo:
- **CrewAI**: Quando você quer montar um "time" de agentes rapidamente
- **Agno**: Quando quer um assistente com ferramentas de forma simples
- **LangGraph**: Quando precisa de **controle total**, **persistência**, **human-in-the-loop**, e **orquestração complexa**

---

## 🏆 Exercício Final: Agente de Análise de Empresa

Construa um agente LangGraph que:

1. **Receba** o nome de uma empresa
2. **Nó `researcher`**: Busca informações na web sobre a empresa
3. **Nó `analyst`**: Analisa os dados e gera um score de 0 a 10
4. **Roteamento condicional**: Se o score < 5, volta ao `researcher` para buscar mais dados
5. **Nó `writer`**: Gera o relatório final da empresa
6. **Use checkpointer** para manter o estado entre execuções

### Dicas:
- Use um `TypedDict` com campos: `messages`, `company_name`, `research_data`, `score`, `iteration`
- Use `MemorySaver()` como checkpointer
- Limite a 3 iterações máximo
- Visualize o grafo com `draw_mermaid_png()`

### Esqueleto:

In [ ]:
# ===== SEU CÓDIGO AQUI =====

# 1. Defina o Estado
# class CompanyAnalysisState(TypedDict):
#     ...

# 2. Defina os Nós
# def researcher_node(state): ...
# def analyst_node(state): ...
# def writer_node(state): ...

# 3. Defina as funções de roteamento
# def analyst_decision(state): ...

# 4. Monte o Grafo
# workflow = StateGraph(CompanyAnalysisState)
# ...

# 5. Compile com checkpointer
# memory = MemorySaver()
# app = workflow.compile(checkpointer=memory)

# 6. Execute
# config = {"configurable": {"thread_id": "analise-petrobras"}}
# result = app.invoke({"messages": [HumanMessage(content="Petrobras")], ...}, config)

print("🛠️  Complete o exercício acima!")

---

## Recapitulando o que aprendemos:

| # | Conceito | O que faz |
|---|----------|----------|
| 1 | **Estado (State)** | Dados que passam entre nós |
| 2 | **Nós (Nodes)** | Funções que processam o estado |
| 3 | **Arestas (Edges)** | Caminhos entre nós (fixos ou condicionais) |
| 4 | **Checkpointers** | Persistem estado entre execuções |
| 5 | **Human-in-the-Loop** | Pausa, inspeciona e modifica antes de continuar |
| 6 | **Streaming** | Tokens em tempo real para UX responsiva |
| 7 | **Time Travel** | Replay e fork de estados anteriores |
| 8 | **Multi-Nó** | Pipelines com lógica complexa e loops |
| 9 | **Supervisor** | Orquestração dinâmica de múltiplos agentes |
| 10 | **Reducers** | Controle fino de como o estado é atualizado |